In [10]:
import os
import sys
import glob
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option('display.max_colwidth', 150)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib as mpl
import seaborn as sns
from math import log
from scipy import stats
sns.set_theme()
try:
    from cellacdc import cca_functions
    from cellacdc import myutils
except FileNotFoundError:
    # Check if user has developer version --> add the Cell_ACDC/cellacdc
    # folder to path and import from thre
    sys.path.insert(0, '../cellacdc/')
    from cellacdc import cca_functions
    from cellacdc import myutils

# ***import df saved at the end of script3*** 

In [11]:
#for computer
overall_df = pd.read_csv(r"Y:\dfs_for_final_code\20231013_20231019_repscombined\reps_combined_fluorescence_analysis_dataset.csv")

C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_17504\1627219939.py:2: DtypeWarning: Columns (29,30,51,116) have mixed types. Specify dtype option on import or set low_memory=False.
  overall_df = pd.read_csv(r"Y:\dfs_for_final_code\20231013_20231019_repscombined\reps_combined_fluorescence_analysis_dataset.csv")


# ***calculate cytoplasm metrics*** 

In [12]:
# calculate combined_mother_bud_area_pixels_cytoplasm
# combined_mother_bud_area_pixels_cytoplasm = combined_mother_bud_area_pixels_cell - combined_mother_bud_area_pixels_nuc


overall_df['combined_mother_bud_area_pixels_cytoplasm'] = overall_df.apply(
    lambda x: x.loc['combined_mother_bud_area_pixels_cell']-x.loc['combined_mother_bud_area_pixels_nuc'],
    axis=1
)

#for each channel
    # calculate combined_mother_bud_sum_pixel_intensity_cytoplasm
    # combined_mother_bud_sum_pixel_intensity_cytoplasm = combined_mother_bud_sum_pixel_intensity_cell - combined_mother_bud_sum_pixel_intensity_nuc
    
    # calculate combined_mother_bud_mean_pixel_intensity_cytoplasm
    # combined_mother_bud_mean_pixel_intensity_cytoplasm = combined_mother_bud_sum_pixel_intensity_cytoplasm/combined_mother_bud_area_pixels_cytoplasm
    
    # calculate combined_mother_bud_median_pixel_intensity_cytoplasm
    # combined_mother_bud_median_pixel_intensity_cytoplasm = combined_mother_bud_median_pixel_intensity_cell - combined_mother_bud_median_pixel_intensity_nuc

channels = ['mCitrine_Yagya', 'mScarlet_Ph3_Yagya', 'Dia_Ph3_mCherry']

for ch in channels:
    try:
        #for cytoplasm compartment, calculate combined_mother_bud_sum_pixel_intensity_cytoplasm
        overall_df[f'{ch}_combined_mother_bud_sum_pixel_intensity_cytoplasm'] = overall_df.apply(
            lambda x: (
                x.loc[f'{ch}_combined_mother_bud_sum_pixel_intensity_cell'] - x.loc[f'{ch}_combined_mother_bud_sum_pixel_intensity_nuc']
                ),
            axis=1
        )
        #for cytoplasm compartment, calculate combined_mother_bud_mean_pixel_intensity_cytoplasm
        overall_df[f'{ch}_combined_mother_bud_mean_pixel_intensity_cytoplasm'] = overall_df.apply(
            lambda x: (
                x.loc[f'{ch}_combined_mother_bud_sum_pixel_intensity_cytoplasm'] / x.loc['combined_mother_bud_area_pixels_cytoplasm']
                ),
            axis=1
        )
        #for cytoplasm compartment, calculate combined_mother_bud_median_pixel_intensity_cytoplasm
        overall_df[f'{ch}_combined_mother_bud_median_pixel_intensity_cytoplasm'] = overall_df.apply(
            lambda x: (
                x.loc[f'{ch}_combined_mother_bud_median_pixel_intensity_cell'] - x.loc[f'{ch}_combined_mother_bud_median_pixel_intensity_nuc']
                ),
            axis=1
        )
        
        
    except KeyError:
        continue


# ***calculate fluorescence concentrations*** 

In [13]:

for ch in channels:
    try:
        #for cell compartment, calculate combined_conc_mother_bud_cell
        overall_df[f'{ch}_combined_conc_mother_bud_cell'] = overall_df.apply(
            lambda x: (
                x.loc[f'{ch}_combined_amount_mother_bud_cell'] / x.loc['combined_mother_bud_volume_cell']
                ),
            axis=1
        )
        #for nuclear compartment, calculate combined_conc_mother_bud_nuc
        overall_df[f'{ch}_combined_conc_mother_bud_nuc'] = overall_df.apply(
            lambda x: (
                x.loc[f'{ch}_combined_amount_mother_bud_nuc'] / x.loc['combined_mother_bud_volume_nuc']
                ),
            axis=1
        )
        #for cytoplasm compartment, calculate combined_amount_mother_bud_cytoplasm
        overall_df[f'{ch}_combined_amount_mother_bud_cytoplasm'] = overall_df.apply(
            lambda x: (
                x.loc[f'{ch}_combined_amount_mother_bud_cell'] - x.loc[f'{ch}_combined_amount_mother_bud_nuc']
                ),
            axis=1
        )
        #for cytoplasm compartment, calculate combined_conc_mother_bud_cytoplasm
        overall_df[f'{ch}_combined_conc_mother_bud_cytoplasm'] = overall_df.apply(
            lambda x: (
                x.loc[f'{ch}_combined_amount_mother_bud_cytoplasm'] / x.loc['combined_mother_bud_volume_cytoplasm']
                ),
            axis=1
        )
        
        
    except KeyError:
        continue


# ***subtract cytoplasmic mean mCitrine pixel intensity from nuclear mean mCitrine pixel intensity*** 

In [14]:

overall_df['mCitrine_Yagya_combined_mother_bud_mean_pixel_intensity_nuc_cytosubtracted'] = overall_df.apply(
    lambda x: x.loc['mCitrine_Yagya_combined_mother_bud_mean_pixel_intensity_nuc']-x.loc['mCitrine_Yagya_combined_mother_bud_mean_pixel_intensity_cytoplasm'],
    axis=1
)


# ***for all cells in G1, set _rel columns to np.nan*** 

In [15]:
rel_cols = overall_df.filter(regex='_rel$').columns.tolist()

In [16]:
for column in overall_df.columns:
    if column in rel_cols:
        overall_df[column][overall_df.cell_cycle_stage_sa == "G1"] = np.nan

        

C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_17504\1379762205.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  overall_df[column][overall_df.cell_cycle_stage_sa == "G1"] = np.nan
C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_17504\1379762205.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  overall_df[column][overall_df.cell_cycle_stage_sa == "G1"] = np.nan
C:\Users\yagya.chadha\AppData\Local\Temp\ipykernel_17504\1379762205.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/u

# ***update f_pos_cell_id (unique cycle identifier) with rep info*** 

In [17]:
overall_df['rep_str_pos_cell_id_sa'] = overall_df.apply(lambda x: f'{x["f_pos_cell_id_sa"]}_Rep_{int(x["rep"])}', axis=1)
overall_df = overall_df.drop(['f_pos_cell_id_sa'], axis=1)
overall_df = overall_df.rename(columns={"rep_str_pos_cell_id_sa": "f_pos_cell_id_sa"})

In [18]:
overall_df.to_csv(r"Y:\dfs_for_final_code\20231013_20231019_repscombined\final_processed_reps_combined_fluorescence_analysis_dataset.csv")